# Wilks / coverage analysis -- frequentist Delta-chi2 interval calibration

Analyzes the output of `scan/wilks.py` (Workstream E): are the profile-likelihood
Delta-chi2 intervals the tool reports actually calibrated? For each parameter of
interest (POI), the per-mock test statistic is

    t = chi2_cond(POI = truth) - chi2_global .

Under Wilks' theorem {t} ~ chi2_1, so the empirical coverage frac(t<1)=0.683 (1 sigma)
and frac(t<3.84)=0.95 (2 sigma) is the deliverable (review gap #4).

**Inputs** (in `scan/results/`), produced by jobs 55202438 (lcdm) / 55202439 (lcdm_neff):
- `wilks_<config>_merged.npz` -- pooled across ranks by `wilks_collect.py` (preferred).
- `wilks_<config>_r*.npz` -- per-rank slices, checkpointed after each POI (fallback,
  and what lets this notebook show a partially finished run).

**Dependencies:** numpy, scipy, matplotlib only -- no GPU / ABCMB. Any kernel with the
SciPy stack works (e.g. the `actdr6` kernel on NERSC JupyterHub). Just run all cells;
re-run to refresh while a job is still going.


In [ ]:
import os, glob
import numpy as np
from scipy.stats import chi2 as chi2dist, kstest, norm as normdist
import matplotlib.pyplot as plt
%matplotlib inline

# Where the result files live (edit if you copied them elsewhere):
RESDIR = '/pscratch/sd/c/carag/ABCMB-k/scan/results'

# config file-stem -> display label
CONFIGS = {'lcdm': 'LCDM (6 POI)', 'lcdm_neff': 'LCDM + Neff (7 POI)'}

CHI2_1_MEDIAN = 0.4549364231
LVL1, LVL2 = 1.0, 3.8414588206941          # Delta-chi2 levels for 1sigma / 2sigma
NOM = {LVL1: 0.6826894921, LVL2: 0.95}     # nominal coverage of chi2_1 at those levels

print('results dir:', RESDIR)
print('wilks files present:')
for f in sorted(glob.glob(os.path.join(RESDIR, 'wilks_*.npz'))):
    print('   ', os.path.basename(f))


In [ ]:
def load_results(key, resdir=RESDIR):
    '''Pooled results for one config. Prefers wilks_<key>_merged.npz; else pools the
    per-rank wilks_<key>_r*.npz slices. Robust to a partial run: a POI is included if
    present in any slice, pooled across the slices that have it.'''
    merged = os.path.join(resdir, 'wilks_%s_merged.npz' % key)
    slices = sorted(glob.glob(os.path.join(resdir, 'wilks_%s_r*.npz' % key)))
    single = os.path.join(resdir, 'wilks_%s.npz' % key)
    out = dict(t={}, z={}, cert={})
    if os.path.exists(merged):
        src_files = [merged]; out['source'] = os.path.basename(merged)
    elif slices:
        src_files = slices; out['source'] = '%d per-rank slice(s)' % len(slices)
    elif os.path.exists(single):
        src_files = [single]; out['source'] = os.path.basename(single)
    else:
        return None
    d0 = np.load(src_files[0], allow_pickle=True)
    pois = [str(p) for p in d0['pois']]
    out['order'] = [str(p) for p in d0['order']]
    out['theta_true'] = np.asarray(d0['theta_true'], float)
    cg, gg = [], []
    tp = {p: [] for p in pois}; zp = {p: [] for p in pois}
    ct = {p: ([], []) for p in pois}
    for f in src_files:
        d = np.load(f, allow_pickle=True)
        cg.append(np.asarray(d['chi2_global'], float))
        gg.append(np.asarray(d['gnorm_global'], float))
        for p in pois:
            if ('t_%s' % p) in d.files:
                tp[p].append(np.asarray(d['t_%s' % p], float))
                zp[p].append(np.asarray(d['z_%s' % p], float))
                if ('cert_tshared_%s' % p) in d.files:
                    ct[p][0].append(np.asarray(d['cert_tshared_%s' % p], float))
                    ct[p][1].append(np.asarray(d['cert_texact_%s' % p], float))
    out['chi2_global'] = np.concatenate(cg)
    out['gnorm_global'] = np.concatenate(gg)
    present = []
    for p in pois:
        if tp[p]:
            out['t'][p] = np.concatenate(tp[p]); out['z'][p] = np.concatenate(zp[p])
            present.append(p)
            if ct[p][0]:
                out['cert'][p] = (np.concatenate(ct[p][0]), np.concatenate(ct[p][1]))
    out['pois'] = present
    out['n'] = len(out['chi2_global'])
    return out


def coverage_stats(t):
    '''Coverage at the chi2_1 levels (binomial error + deviation), KS, moments.
    Coverage uses raw t: a (rare) t<0 means chi2_cond < chi2_global, i.e. truth well
    inside -> correctly counts as covered. neg is reported separately.'''
    t = np.asarray(t, float); t = t[np.isfinite(t)]; n = max(len(t), 1)
    res = dict(n=len(t), mean=float(t.mean()), median=float(np.median(t)),
               neg=float(np.mean(t < -1e-6)), cov={})
    for lvl in (LVL1, LVL2):
        frac = float(np.mean(t <= lvl))
        err = float(np.sqrt(max(frac * (1 - frac) / n, 1e-12)))
        nom = NOM[lvl]
        se_nom = float(np.sqrt(max(nom * (1 - nom) / n, 1e-12)))
        res['cov'][lvl] = dict(frac=frac, err=err, nom=nom, dev=(frac - nom) / se_nom)
    ks = kstest(t, chi2dist(1).cdf)
    res['ks_p'] = float(ks.pvalue); res['ks_stat'] = float(ks.statistic)
    return res


In [ ]:
def show_table(key, R):
    label = CONFIGS.get(key, key)
    if R is None:
        print('=== %s (%s): NO RESULTS YET ===' % (label, key))
        print('    (no wilks_%s_*.npz in the results dir)' % key)
        print('')
        return None
    print('================  %s  (%s)  ================' % (label, key))
    print('source: %s    pooled mocks: %d    POIs: %d/%d'
          % (R['source'], R['n'], len(R['pois']), len(R['order'])))
    missing = [p for p in R['order'] if p not in R['pois']]
    if missing:
        print('    still missing (run not finished): %s' % missing)
    print('global-fit max||g||: %.2e' % np.max(R['gnorm_global']))
    print('%10s %5s %6s %7s %22s %22s %7s %5s'
          % ('POI', 'n', 'mean', 'median', 'cov(t<1)', 'cov(t<3.84)', 'KS p', 'neg'))
    stats = {}
    for p in R['pois']:
        s = coverage_stats(R['t'][p]); stats[p] = s
        c1 = s['cov'][LVL1]; c2 = s['cov'][LVL2]
        cov1 = '%5.1f+-%.1f%% (%+.1fsig)' % (c1['frac']*100, c1['err']*100, c1['dev'])
        cov2 = '%5.1f+-%.1f%% (%+.1fsig)' % (c2['frac']*100, c2['err']*100, c2['dev'])
        print('%10s %5d %6.3f %7.3f %22s %22s %7.3f %4.0f%%'
              % (p, s['n'], s['mean'], s['median'], cov1, cov2, s['ks_p'], s['neg']*100))
    print('nominal: median %.3f, cov(t<1)=68.3%%, cov(t<3.84)=95.0%%' % CHI2_1_MEDIAN)
    if R['pois']:
        c1s = [stats[p]['cov'][LVL1]['frac'] for p in R['pois']]
        c2s = [stats[p]['cov'][LVL2]['frac'] for p in R['pois']]
        print('')
        print('HEADLINE: empirical coverage is %.0f%% (1sigma) / %.0f%% (2sigma) '
              'vs nominal 68.3%% / 95.0%% over %d mocks'
              % (np.mean(c1s)*100, np.mean(c2s)*100, R['n']))
        print('          per-POI 1sigma range %.0f-%.0f%%, 2sigma range %.0f-%.0f%%'
              % (min(c1s)*100, max(c1s)*100, min(c2s)*100, max(c2s)*100))
    print('')
    return stats

ALL, STATS = {}, {}
for key in CONFIGS:
    ALL[key] = load_results(key)
    STATS[key] = show_table(key, ALL[key])


## Per-POI diagnostic plots

For each POI: (left) the test-statistic histogram vs the chi2_1 pdf; (middle) the
empirical CDF vs chi2_1 with the 1 sigma / 2 sigma levels marked; (right) the signed
pull `sign(theta_hat - theta_true) * sqrt(t)` vs N(0,1) (for a Gaussian, t = z^2, so a
sigma-free calibration check).


In [ ]:
def plot_poi(p, t, z):
    t = np.asarray(t, float); t = t[np.isfinite(t)]
    z = np.asarray(z, float); z = z[np.isfinite(z)]
    tc = np.clip(t, 0, None); hi = max(10.0, np.percentile(tc, 99))
    xs = np.linspace(1e-3, hi, 400)
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
    ax[0].hist(tc, bins=40, range=(0, hi), density=True, alpha=0.6, label='mocks')
    ax[0].plot(xs, chi2dist(1).pdf(xs), 'r-', lw=2, label='chi2_1')
    ax[0].set_xlabel('t = chi2_cond(truth) - chi2_global'); ax[0].set_ylabel('pdf')
    ax[0].set_title('%s: test statistic' % p); ax[0].legend()
    ts = np.sort(tc); ecdf = np.arange(1, len(ts) + 1) / len(ts)
    ax[1].plot(ts, ecdf, drawstyle='steps-post', label='empirical')
    ax[1].plot(xs, chi2dist(1).cdf(xs), 'r-', lw=2, label='chi2_1')
    ax[1].axvline(LVL1, ls=':', lw=0.8, color='gray')
    ax[1].axvline(LVL2, ls=':', lw=0.8, color='gray')
    ax[1].set_xlabel('t'); ax[1].set_ylabel('CDF'); ax[1].set_title('coverage'); ax[1].legend()
    ax[2].hist(z, bins=40, range=(-4, 4), density=True, alpha=0.6, label='mocks')
    zs = np.linspace(-4, 4, 400)
    ax[2].plot(zs, normdist.pdf(zs), 'r-', lw=2, label='N(0,1)')
    ax[2].set_xlabel('pull (theta_hat - theta_true)/sigma')
    ax[2].set_title('%s: pull' % p); ax[2].legend()
    fig.tight_layout(); plt.show()

for key, R in ALL.items():
    if R is None:
        continue
    print('### %s ###' % CONFIGS.get(key, key))
    for p in R['pois']:
        plot_poi(p, R['t'][p], R['z'][p])


## Summary: coverage vs nominal, all POIs


In [ ]:
for key, R in ALL.items():
    if R is None or STATS[key] is None or not R['pois']:
        continue
    pois = R['pois']; x = np.arange(len(pois))
    fig, ax = plt.subplots(figsize=(1.3 * len(pois) + 3, 4))
    for lvl, col, lab in [(LVL1, 'C2', '68.3% (t<1)'), (LVL2, 'C3', '95% (t<3.84)')]:
        fr = np.array([STATS[key][p]['cov'][lvl]['frac'] for p in pois])
        er = np.array([STATS[key][p]['cov'][lvl]['err'] for p in pois])
        ax.errorbar(x, fr, yerr=er, fmt='o-', color=col, capsize=3, label=lab)
        ax.axhline(NOM[lvl], ls='--', lw=0.8, color=col)
    ax.set_xticks(x); ax.set_xticklabels(pois, rotation=30, ha='right')
    ax.set_ylabel('empirical coverage'); ax.set_ylim(0.4, 1.0)
    ax.set_title('%s: Wilks coverage vs nominal (n=%d)' % (CONFIGS.get(key, key), R['n']))
    ax.legend(); fig.tight_layout(); plt.show()


## Gate (b): shared-Jacobian vs exact per-mock-Jacobian certificate

The production runs also re-fit a `WK_VALIDATE`-sized subsample with an exact per-mock
Jacobian. `max|dt| << 0.1` certifies the cheap shared-Jacobian fitter reproduces the
exact fixed point (so the coverage above is trustworthy). Empty until a run reaches the
cert stage (it runs after all conditional fits).


In [ ]:
shown = False
for key, R in ALL.items():
    if R is None or not R['cert']:
        continue
    shown = True
    print('=== %s: per-mock-Jacobian certificate (gate b) ===' % CONFIGS.get(key, key))
    for p, (tsh, tex) in R['cert'].items():
        dt = np.abs(np.asarray(tsh, float) - np.asarray(tex, float))
        print('   %10s : max|dt|=%.3f  med|dt|=%.3f  (n=%d)'
              % (p, dt.max(), np.median(dt), len(dt)))
    print('')
if not shown:
    print('No cert arrays present yet (the cert runs after all conditional fits, WK_VALIDATE>0).')


## Reading the results

- **Coverage is the deliverable.** `cov(t<1)` should be ~ 68.3% and `cov(t<3.84)` ~ 95%.
  The `+-` is the binomial error on the measurement; `(sig)` is the deviation from nominal
  (|dev| <~ 2 sig is consistent at n=500). The per-POI `median` should sit near 0.455.
- **Negative t (`neg`).** A few percent of mocks can have t slightly < 0 (chi2_cond just
  below chi2_global) -- the Gauss-Newton roughness floor at l=2508, not a calibration
  failure. Those mocks ARE covered (truth well inside), so they count correctly toward
  cov(t<1). A *large* neg fraction (>> 10%) would instead flag global-fit under-convergence.
- **KS p-value** tests the whole {t} against chi2_1; a small p with good coverage usually
  just reflects that floor/the negatives in the bulk, not interval miscalibration.
- **Gate (b) cert:** max|dt| << 0.1 confirms the cheap shared-J fitter == the exact fixed point.

### For reference -- gate (a) preview from the first (timed-out) run, N=500
These per-rank medians already showed good calibration (vs chi2_1 median 0.455):
- LCDM: h 0.50/0.48, omega_b 0.50/0.55, omega_cdm 0.46/0.47, n_s 0.47/0.42
- LCDM+Neff: h 0.30/0.28, omega_b 0.41/0.30, omega_cdm 0.43/0.52

### If a run is still going / was interrupted
The per-rank `wilks_<config>_r*.npz` are checkpointed after each POI, so re-running this
notebook shows whatever POIs have finished (the table flags any still missing). The merged
npz + PNGs land once the job's trailing `wilks_collect.py` runs. To regenerate them by hand:
`python scan/wilks_collect.py lcdm` (and `lcdm_neff`).
